In [2]:
import re
from typing import Dict, Any, List
import pprint

import manim as mn
from manim import *

In [3]:
config.verbosity = "WARNING"
config.background_color = BLUE_A
Text.set_default(color=BLACK, stroke_color=BLACK)
MathTex.set_default(color=BLACK, stroke_color=BLACK)

In [4]:
#pokemon name, -form(optional), (gender)(optional), @ item(optional)
pokemon_item_re = re.compile(r'^\s*([A-Za-z0-9\-\s]+?)(?:-(\S+))?(?:\s+\((M|F|N|Genderless)\))?\s*(?:@\s*(.+))?\s*$', re.I)
#groups 1+ letter, :, (.+)rest of line
ability_extras_re = re.compile(r'^\s*([A-Za-z]+):\s*(.+)$', re.I)
#groups of 1+ digit, 0+ whitespace, 1-3 letter stat
evs_re = re.compile(r'(\d+)\s*([A-Za-z]{1,3})')
#groups of whitespace - whitespace, rest of line
moves_re = re.compile(r'^\s*-\s*(.+)$')


def parse_evs(evs_str: str) -> Dict[str, int]:
    evs = {}
    for match in evs_re.finditer(evs_str):
        val, stat = match.groups()
        stat_map = {'HP': 'hp', 'Atk': 'atk', 'Def': 'def', 'SpA': 'spa', 'SpD': 'spd', 'Spe': 'spe',
                    'Spd': 'spd', 'Sp': 'spd'}  # include some variants
        key = stat_map.get(stat, stat.lower())
        evs[key] = int(val)
    return evs


def parse_pokemon_set(text: str) -> Dict[str, Any]:
    """
    Parse a single Pokemon set in common showdown-style format into a dict.
    Returns keys: name, form (optional), gender (optional), item (optional),
    ability (optional), level (int, optional), shiny (bool), evs (dict),
    nature (optional), moves (list).
    """
    lines = [l.rstrip() for l in text.strip().splitlines() if l.strip() != '']
    if not lines:
        return {}

    result: Dict[str, Any] = {
        'name': None,
        'form': None,
        'gender': None,
        'item': None,
        'ability': None,
        'level': None,
        'shiny': False,
        'evs': {},
        'nature': None,
        'moves': []
    }

    # Header line (first non-empty line)
    header = lines[0]
    m = pokemon_item_re.match(header)
    if m:
        name, form, gender, item = m.groups()
        result['name'] = name.strip()
        if form:
            result['form'] = form.strip()
        if gender:
            g = gender.strip()
            result['gender'] = g
        if item:
            result['item'] = item.strip()

    # parse remaining lines
    for line in lines[1:]:
        # moves
        moves = moves_re.match(line)
        if moves:
            move = moves.group(1).strip()
            result['moves'].append(move)
            continue

        # key: value lines
        extras = ability_extras_re.match(line)
        if extras:
            key, val = extras.groups()
            key_lower = key.lower()
            if key_lower == 'ability':
                result['ability'] = val.strip()
            elif key_lower == 'level':
                try:
                    result['level'] = int(val.strip())
                except ValueError:
                    result['level'] = 50
            elif key_lower == 'shiny':
                result['shiny'] = True
            elif key_lower == 'evs':
                result['evs'] = parse_evs(val)
            else:
                # fallback: store unknown key
                result[key_lower] = val.strip()
            continue

        # nature line (ends with "Nature")
        if line.strip().endswith('Nature'):
            # example "Jolly Nature"
            parts = line.strip().split()
            if parts:
                result['nature'] = parts[0].strip()
            continue


    return result

def parse_pokemon_team(text: str) -> list[dict[str, Any]]:
    sets = text.strip().split('\n\n')
    return [parse_pokemon_set(s) for s in sets]


# Example usage
if __name__ == '__main__':
    sample = """
Salamence-Mega (M) @ Salamencite  
Ability: Intimidate  
Level: 50  
EVs: 2 HP / 32 Atk / 32 Spe  
Adamant Nature  
- Double-Edge  
- Roost  
- Tailwind  
- Protect  

Indeedee-F (F) @ Psychic Seed  
Ability: Psychic Surge  
Level: 50  
EVs: 32 HP / 19 Def / 15 SpD  
Calm Nature  
- Trick Room  
- Follow Me  
- Psychic  
- Hyper Voice  
"""
    
    parsed = parse_pokemon_team(sample)
    pprint.pprint(parsed)

[{'ability': 'Intimidate',
  'evs': {'atk': 32, 'hp': 2, 'spe': 32},
  'form': 'Mega',
  'gender': 'M',
  'item': 'Salamencite',
  'level': 50,
  'moves': ['Double-Edge', 'Roost', 'Tailwind', 'Protect'],
  'name': 'Salamence',
  'nature': 'Adamant',
  'shiny': False},
 {'ability': 'Psychic Surge',
  'evs': {'def': 19, 'hp': 32, 'spd': 15},
  'form': 'F',
  'gender': 'F',
  'item': 'Psychic Seed',
  'level': 50,
  'moves': ['Trick Room', 'Follow Me', 'Psychic', 'Hyper Voice'],
  'name': 'Indeedee',
  'nature': 'Calm',
  'shiny': False}]


In [5]:
%%manim -qm DamageFormula

class DamageFormula(Scene):
    def construct(self):
        # build first line as three pieces (start uncolored)
        left = Text("Salamence-Mega (M)", font_size=36, color=BLACK)
        at   = Text(" @ ", font_size=36, color=BLACK)
        right= Text("Salamencite", font_size=36, color=BLACK)
        first = VGroup(left, at, right).arrange(RIGHT, buff=0.06)

        # build the rest
        rest_block = Paragraph(
            "Ability: Intimidate",
            "Level: 50",
            "Shiny: Yes",
            "EVs: 5 HP / 32 Atk / 7 Def / 3 SpA / 19 Spe",
            "Naughty Nature",
            "- Double-Edge",
            "- Hyper Voice",
            "- Tailwind",
            "- Protect",
            line_spacing=0.35
        ).scale(0.7)

        # combine and add once
        group = VGroup(first, rest_block).arrange(DOWN, aligned_edge=LEFT)
        group.to_edge(LEFT, buff=1)
        self.add(group)

        # animate the rest lines falling (keep first in place)
        rest = rest_block  # Paragraph returns submobjects for each line
        down_shift = config.frame_height * DOWN
        animations = [line.animate.shift(down_shift) for line in rest]
        self.play(*animations, run_time=1.5)
        self.wait()

        # create each regex line (as you already do)
        regex_line_1 = Text("([A-Za-z0-9\\-\\s]+?)", font_size=24, color=RED)
        regex_line_2 = Text("(@)",                          font_size=24, color=YELLOW)
        regex_line_3 = Text("([A-Za-z0-9\\-\\s]+?)", font_size=24, color=GREEN)

        # stack them vertically instead of side-by-side
        regex_group = VGroup(regex_line_1, regex_line_2, regex_line_3).arrange(DOWN, buff=0.06, aligned_edge=LEFT)

        # place the stacked group under the first name part
        regex_group.next_to(first, DOWN, buff=0.15)
        regex_group.align_to(first, LEFT)

        self.play(
            Write(regex_group, run_time=0.6),
            left.animate.set_color(RED),
            at.animate.set_color(YELLOW),
            right.animate.set_color(GREEN),
        )
        self.wait()

Manim Community v0.20.1